## Artifact
Library Import
All required libraries for data handling, file management, numerical operations, and ML model persistence were successfully imported.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

## Loading the Processed ML Data
I loaded the processed training and testing features along with their corresponding target values. I converted the target data into Series format and removed any accidental index columns from the feature datasets. Finally, I checked the dimensions to confirm that the data was loaded correctly and is ready for model training.

In [ ]:
X_train = pd.read_csv("X_train_processed.csv")
X_test = pd.read_csv("X_test_processed.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

# Remove accidental index columns
X_train = X_train.loc[
    :, ~X_train.columns.str.contains("^Unnamed")
]

X_test = X_test.loc[
    :, ~X_test.columns.str.contains("^Unnamed")
]

print("Data loaded successfully!")

print("\nX_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


## Loading the Saved Best Model
I checked whether my saved best model was available before loading it. If the file was missing, the code provided a clear error message. Once found, the trained model was loaded using Joblib, and its model type was displayed to verify that the correct model was successfully retrieved

In [ ]:
model_file = "ev_sales_best_model.pkl"

if not os.path.exists(model_file):

    raise FileNotFoundError(
        f"{model_file} was not found. "
        "Run the final model save cell in "
        "04_Model_Building.ipynb first."
    )

final_model = joblib.load(
    model_file
)

print("Final model loaded successfully!")

print(
    "Model type:",
    type(final_model).__name__
)

## Test Predictions
After loading the best-performing model, I used it to predict EV sales for the unseen testing data. I stored the predictions in test_predictions and displayed the first ten results as an initial verification. These predictions will be compared with the actual EV sales values to measure the model's performance


In [ ]:
test_predictions = final_model.predict(
    X_test
)

print("Prediction successful!")

print("\nFirst 10 predictions:")

print(
    test_predictions[:10]
)

## Model Performance Evaluation
I evaluated my final model using four regression metrics: MAE, MSE, RMSE, and R². MAE, MSE, and RMSE measure the prediction error, while R² measures how well the model explains the variation in EV sales. Using multiple metrics gives a more complete understanding of the model's performance.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test,
    test_predictions
)

mse = mean_squared_error(
    y_test,
    test_predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    test_predictions
)


print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R2  :", round(r2, 4))

## Creating the Prediction Artifact
I created a prediction dataset containing the actual EV sales, predicted EV sales, and the difference between them. This allows me to compare the model's predictions with the actual values and analyse prediction errors. I then saved these results as a CSV file for further analysis and reporting.

In [ ]:
prediction_df = pd.DataFrame({

    "Actual_EV_Sales":
        y_test.values,

    "Predicted_EV_Sales":
        test_predictions,

    "Error":
        y_test.values - test_predictions

})

prediction_df.to_csv(
    "ev_sales_predictions.csv",
    index=False
)

print(
    "Prediction artifact saved successfully!"
)

print(
    "File: ev_sales_predictions.csv"
)

## Saving Feature Names

I saved the final processed feature names in a JSON file. This is important because future prediction data must contain the same features in the same structure as the data used to train the model. I also checked the total number of features to verify the model input structure.

In [ ]:
feature_names = X_train.columns.tolist()

with open(
    "feature_names.json",
    "w"
) as file:

    json.dump(
        feature_names,
        file,
        indent=4
    )

print("Feature names saved successfully!")

print(
    "Number of features:",
    len(feature_names)
)

## Feature Importance Analysis
analysed the importance of the input features used by my final model. The features were ranked from highest to lowest importance, and the top 15 were displayed. This helps identify which variables have the greatest influence on the model's EV-sales predictions.

In [ ]:
if hasattr(
    final_model,
    "feature_importances_"
):

    feature_importance = pd.DataFrame({

        "Feature":
            X_train.columns,

        "Importance":
            final_model.feature_importances_

    })

    feature_importance = feature_importance.sort_values(
        by="Importance",
        ascending=False
    )

    display(
        feature_importance.head(15)
    )

else:

    print(
        "This model does not provide "
        "feature_importances_."
    )

## Saving Feature Importance

After identifying the most important features, I saved the feature-importance results as a CSV file. This allows me to reuse the results for model interpretation, visualization, and project reporting without having to calculate them again.

In [ ]:
if "feature_importance" in globals():

    feature_importance.to_csv(
        "feature_importance.csv",
        index=False
    )

    print(
        "Feature importance saved successfully!"
    )

## Saving Model Metrics
I created a model-metrics file containing the important details of my final model, including the target variable, model type, number of features, training and testing samples, and evaluation metrics. I saved this information in JSON format so it can be easily reused for reporting and future model analysis.

In [ ]:
metrics = {

    "target": "ev_sales",

    "model": type(final_model).__name__,

    "number_of_features":
        len(X_train.columns),

    "training_samples":
        len(X_train),

    "testing_samples":
        len(X_test),

    "MAE":
        float(mae),

    "MSE":
        float(mse),

    "RMSE":
        float(rmse),

    "R2_Score":
        float(r2)
}

with open(
    "model_metrics.json",
    "w"
) as file:

    json.dump(
        metrics,
        file,
        indent=4
    )

print(
    "Model metrics saved successfully!"
)

print(
    json.dumps(
        metrics,
        indent=4
    )
)

## Creating the Artifact Folder
I created a separate artifacts folder to organize all the important outputs generated by my machine-learning project, such as the trained model, predictions, feature names, feature importance, and model metrics. I also displayed the absolute path to verify where the folder was created.

In [ ]:

artifact_folder = "artifacts"

os.makedirs(
    artifact_folder,
    exist_ok=True
)

print(
    "Artifact folder created:"
)

print(
    os.path.abspath(artifact_folder)
)

## Copying ML Artifacts into the Artifact Folder

I collected all the important outputs from my machine-learning workflow and copied them into a dedicated artifacts folder. Before copying, I checked whether each file existed. This keeps the final model and its supporting files organized in one location and makes them easier to reuse.

In [ ]:

import shutil

files_to_copy = [

    "ev_sales_best_model.pkl",

    "ev_sales_predictions.csv",

    "feature_names.json",

    "model_metrics.json",

    "feature_importance.csv"
]

for file_name in files_to_copy:

    if os.path.exists(file_name):

        shutil.copy(
            file_name,
            os.path.join(
                artifact_folder,
                file_name
            )
        )

        print(
            "Copied:",
            file_name
        )

    else:

        print(
            "Not found:",
            file_name
        )

print("\nArtifact copying completed!")

## Final Artifact Verification
The final verification confirms that the generated machine-learning artifacts are successfully stored and organized for future use.

In [ ]:
artifact_files = os.listdir(
    artifact_folder
)

for file_name in artifact_files:

    print("✓", file_name)

print(
    "\nTotal artifacts:",
    len(artifact_files)
)

## Final Machine Learning Artifact Conclusion

The machine-learning workflow for predicting **EV sales** was successfully completed. The processed training and testing datasets were loaded, and the final trained model was successfully retrieved from the saved model file.

The model was used to generate EV sales predictions on the unseen test dataset. Its performance was evaluated using **MAE, MSE, RMSE, and R² Score**, providing multiple measures of prediction accuracy and model effectiveness.

The prediction results were saved in `ev_sales_predictions.csv`, containing the actual EV sales, predicted EV sales, and prediction errors. The processed feature names were saved in `feature_names.json`, while the feature-importance results were stored in `feature_importance.csv` when supported by the final model. The model performance information was also saved in `model_metrics.json`.

Finally, all important outputs were organized inside the dedicated **`artifacts`** folder and verified using the final artifact-verification step. This ensures that the trained model and supporting files are available for reuse, reporting, prediction, and potential deployment.

Overall, the machine-learning stage successfully produced a **reusable EV sales prediction model together with the required prediction, feature, evaluation, and model artifacts**. The project is now ready for the next stage, such as **model deployment, prediction interface development, Power BI integration, and final project reporting**.
